# **Smallholder Crop Yield Analysis in Malawi**

This notebook analyzes the impact of climate variability, soil conditions, and adaptation practices on crop yields.
We use advanced machine learning to identify key drivers and predict agricultural productivity.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Load data
df = pd.read_csv('malawi_agriculture_climate_yield_data.csv')
print(f'Data shape: {df.shape}')

### **1. Data Cleaning and Feature Engineering**
We select high-importance features like NDVI and household size while removing local language strings from soil types.
Missing values are filled using median and mode strategies to maintain data integrity.

In [ ]:
target = 'yield_kg_ha'
features = [
    'annual_mean_temperature', 'annual_precipitation_mm', 'precipitation_wettest_quarter_mm',
    'h2018_ndvi_avg', 'h2019_ndvi_avg', 'anntot_avg 2018 to 2019', 
    'soil_type', 'soil_quality', 'slope_pct', 'elevation_m',
    'nutrient_availability', 'nutrient_retention_capacity',
    'irrigation_type', 'organic_fertilizer', 'inorganic_fertilizer',
    'farm_machinery', 'credit_0', 'hhsize', 'erosion_control'
]

df_cleaned = df[[target] + features].copy()
df_cleaned['soil_type'] = df_cleaned['soil_type'].str.split('(').str[0].str.strip()

for col in df_cleaned.columns:
    if df_cleaned[col].dtype == 'object':
        df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].mode()[0])
    else:
        df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].median())

### **2. Outlier Removal and Normalization**
We filter out yield values above the 99th percentile to remove extreme data noise.
A log transformation is applied to the yield variable to reduce skewness and improve model performance.

In [ ]:
limit = df_cleaned['yield_kg_ha'].quantile(0.99)
df_final = df_cleaned[df_cleaned['yield_kg_ha'] <= limit].copy()
df_final['yield_log'] = np.log1p(df_final['yield_kg_ha'])

### **3. Preprocessing and Model Training**
Categorical variables are encoded via One-Hot encoding to handle non-ordinal relationships.
A Random Forest model is then trained on 80% of the data to learn yield patterns.

In [ ]:
X = pd.get_dummies(df_final.drop(['yield_kg_ha', 'yield_log'], axis=1), drop_first=True)
y = df_final['yield_log']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

### **4. Performance Evaluation and Visualization**
The model performance is measured using R-squared and error metrics in original units.
The actual vs predicted plot visualizes how well the model captures the real yield distribution.

In [ ]:
y_pred_log = rf.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_actual = np.expm1(y_test)

print(f'R2 Score: {r2_score(y_test, y_pred_log):.4f}')
print(f'MAE: {mean_absolute_error(y_actual, y_pred):.2f} kg/ha')

plt.figure(figsize=(10, 6))
plt.scatter(y_actual, y_pred, alpha=0.3, color='blue')
plt.plot([y_actual.min(), y_actual.max()], [y_actual.min(), y_actual.max()], 'r--')
plt.xlabel('Actual Yield')
plt.ylabel('Predicted Yield')
plt.title('Actual vs Predicted Yield')
plt.show()

In [ ]:
import pickle

# Save the trained Random Forest model
with open('yield_model.pkl', 'wb') as f:
    pickle.dump(rf, f)

# Save the list of columns to ensure the app matches the model's structure
model_columns = list(X.columns)
with open('model_columns.pkl', 'wb') as f:
    pickle.dump(model_columns, f)